# Physics-Informed Neural Networks for Activity Coefficient Prediction

This notebook presents the development of a physics-informed neural network (PINN) to predict activity coefficients using processed experimental data. By embedding thermodynamic constraints directly into the loss function, we incorporate domain knowledge into the model, ensuring thermodynamic consistency while leveraging a data-driven approach.

## Dependencies

In [ ]:
import copy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import MinMaxScaler
from torchmetrics.regression import R2Score

## PINN modeling

### Designing the Neural Network Components

Define the architecture of the neural network

In [ ]:
torch.manual_seed(42)


class NeuralNetArchitecture(nn.Module):
    def __init__(self, in_dim: int, out_dim: int) -> None:
        super().__init__()
        self.layer_1 = nn.Linear(in_dim, 64)
        self.layer_2 = nn.Linear(64, 32)
        self.layer_3 = nn.Linear(32, 16)
        self.layer_4 = nn.Linear(16, out_dim)

    def forward(self, x):
        x = F.relu(self.layer_1(x))
        x = F.relu(self.layer_2(x))
        x = F.relu(self.layer_3(x))
        x = self.layer_4(x)

        return x

Implement a custom loss function that enforces thermodynamic consistency by embedding the Gibbs-Duhem equation into the standard MSE loss

In [ ]:
def gibbs_duhem_loss(
    y_pred: torch.Tensor, y_true: torch.Tensor, X: torch.Tensor, lambda_gd: int = 1
) -> tuple[torch.Tensor, torch.Tensor]:
    # MSE loss
    mse_loss = F.mse_loss(y_pred, y_true, reduction="mean")

    # Calculate ln(gamma)
    ln_gamma1 = torch.log(y_pred[:, 0])
    ln_gamma2 = torch.log(y_pred[:, 1])

    # Compute partial derivatives
    dgamma1_dx1 = torch.autograd.grad(
        ln_gamma1,
        X,
        grad_outputs=torch.ones_like(ln_gamma1),
        create_graph=True,
        allow_unused=True,
    )[0][:, 2]

    dgamma2_dx1 = torch.autograd.grad(
        ln_gamma2,
        X,
        grad_outputs=torch.ones_like(ln_gamma2),
        create_graph=True,
        allow_unused=True,
    )[0][:, 2]

    # Compute Gibbs-Duhem term
    x1 = X[:, 2]
    x2 = X[:, 5]

    gibbs_duhem_term = x1 * dgamma1_dx1 + x2 * dgamma2_dx1

    # Total loss
    loss = mse_loss + torch.mean(gibbs_duhem_term ** 2) * lambda_gd
    return loss, gibbs_duhem_term

### Reading the data

Defining the data path conveniently:

In [ ]:
DATA_PATH = Path("../data")

Reading the data into a `pandas.DataFrame`:

In [ ]:
data = pd.read_csv(DATA_PATH / "processed" / "toy_problem_input_dataset.csv")

data

Remove rows with NaN values:

In [ ]:
data.dropna(inplace=True)
data.reset_index(drop=True, inplace=True)

data

### PINN training

Defining the device and splitting the data into training and test sets

In [ ]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

features = data.iloc[:, :-2].values
labels = data.iloc[:, -2:].values

X, X_test, y, y_test = train_test_split(
    features, labels, test_size=0.1, random_state=42
)

Defining the cross-validation setup

In [ ]:
k_folds = 10
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

Specifying training hyperparameters

In [ ]:
num_epochs = 20000
learning_rate = 5e-3
lambda_gd = 0
patience = 1000

Executing the cross-validation training loop to evaluate the best selection of hyperparameters

In [ ]:
# Combined loss lists
fold_losses = []
val_losses = []
test_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"Fold {fold + 1}/{k_folds}")

    # Train-validation split for this fold
    X_train_cv, X_val_cv = X[train_idx], X[val_idx]
    y_train_cv, y_val_cv = y[train_idx], y[val_idx]

    # Scale features
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train_cv)
    X_val_scaled = scaler.transform(X_val_cv)
    X_test_scaled = scaler.transform(X_test)

    # Convert to tensors and move to device
    X_train_tensor = torch.tensor(
        X_train_scaled, dtype=torch.float32, requires_grad=True
    ).to(device)
    X_val_tensor = torch.tensor(
        X_val_scaled, dtype=torch.float32, requires_grad=True
    ).to(device)
    X_test_tensor = torch.tensor(
        X_test_scaled, dtype=torch.float32, requires_grad=True
    ).to(device)
    y_train_tensor = torch.tensor(y_train_cv, dtype=torch.float32).to(device)
    y_val_tensor = torch.tensor(y_val_cv, dtype=torch.float32).to(device)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

    # Model and optimizer
    input_size = X_train_cv.shape[1]
    output_size = 2
    model = NeuralNetArchitecture(input_size, output_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Early stopping vars
    best_train_loss = float("inf")
    best_val_loss = float("inf")
    best_model_wts = None
    epochs_no_improve = 0

    # Training loop
    for epoch in range(num_epochs):
        model.train()

        # Forward pass
        outputs = model(X_train_tensor)

        # Loss - Store the physical loss to further evaluation
        combined_loss = gibbs_duhem_loss(outputs, y_train_tensor, X_train_tensor, lambda_gd)
        loss = combined_loss[0]

        # Validation loss
        val_outputs = model(X_val_tensor)
        val_combined_loss = gibbs_duhem_loss(val_outputs, y_val_tensor, X_val_tensor, lambda_gd)
        val_loss = val_combined_loss[0]

        # Backward propagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Early stopping check
        if val_loss < best_val_loss:
            best_train_loss = loss
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

        if (epoch + 1) % 20 == 0:
            print(f"[Fold {fold + 1} | Epoch {epoch + 1}]: Training loss -> {loss}")
            print(
                f"[Fold {fold + 1} | Epoch {epoch + 1}]: Validation loss -> {val_loss}"
            )

    # Evaluate test data
    model.load_state_dict(best_model_wts)
    test_outputs = model(X_test_tensor)
    test_combined_loss = gibbs_duhem_loss(test_outputs, y_test_tensor, X_test_tensor, lambda_gd)
    test_loss = test_combined_loss[0]

    # Store best loss for this fold
    fold_losses.append(best_train_loss.item())
    val_losses.append(best_val_loss.item())
    test_losses.append(test_loss.item())

# Print average loss across folds
print(f"Average loss across {k_folds} folds: {np.mean(fold_losses): .4f}")
print(f"Average validation loss across {k_folds} folds: {np.mean(val_losses): .4f}")
print(f"Average test loss across {k_folds} folds: {np.mean(test_losses): .4f}")

### PINN evaluation

Evaluating the model with the selected hyperparameters

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    features, labels, test_size=0.2, random_state=42
)

# Scale features
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert to tensors and move to device
X_train_tensor = torch.tensor(
    X_train_scaled, dtype=torch.float32, requires_grad=True
).to(device)
X_val_tensor = torch.tensor(
    X_val_scaled, dtype=torch.float32, requires_grad=True
).to(device)
X_test_tensor = torch.tensor(
    X_test_scaled, dtype=torch.float32, requires_grad=True
).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

# Model and optimizer
input_size = X_train.shape[1]
output_size = 2
model = NeuralNetArchitecture(input_size, output_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Early stopping vars
best_train_loss = float("inf")
best_val_loss = float("inf")
best_model_wts = None
epochs_no_improve = 0

# Training loop
for epoch in range(num_epochs):
    model.train()

    # Forward pass
    outputs = model(X_train_tensor)

    # Loss - Store the physical loss to further evaluation
    combined_loss = gibbs_duhem_loss(outputs, y_train_tensor, X_train_tensor, lambda_gd)
    loss = combined_loss[0]

    # Validation loss
    val_outputs = model(X_val_tensor)
    val_combined_loss = gibbs_duhem_loss(val_outputs, y_val_tensor, X_val_tensor, lambda_gd)
    val_loss = val_combined_loss[0]

    # Backward propagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Early stopping check
    if val_loss < best_val_loss:
        best_train_loss = loss
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch + 1}")
        break

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1}: Training loss -> {loss}")
        print(
            f"Epoch {epoch + 1}: Validation loss -> {val_loss}"
        )

# Save best model
MODEL_PATH = Path("./models")
torch.save(best_model_wts, MODEL_PATH / f"trained_pinn.pt")

# Evaluate test data
model.load_state_dict(best_model_wts)
test_outputs = model(X_test_tensor)
test_combined_loss = gibbs_duhem_loss(test_outputs, y_test_tensor, X_test_tensor, lambda_gd)
test_loss = test_combined_loss[0]
test_residuals = test_combined_loss[1]

# Print loss for the best hyperparameter configuration
print(f"Training loss: {best_train_loss.item(): .4f}")
print(f"Validation loss: {best_val_loss.item(): .4f}")
print(f"Testing loss: {test_loss.item(): .4f}")

Plotting Predicted vs Experimental Values for $\gamma_1$

In [ ]:
# Calculate R²
r2_metric = R2Score()
r2 = r2_metric(test_outputs[:, 0], y_test_tensor[:, 0]).item()

# Convert tensors to CPU and NumPy for plotting
y_test_np = y_test_tensor[:, 0].cpu().numpy()
preds_np = test_outputs[:, 0].detach().cpu().numpy()

# Plot the scatter plot
plt.scatter(y_test_np, preds_np, label=f"R² = {r2:.4f}")
plt.xlabel("Experimental values")
plt.ylabel("Predicted values")
plt.axis("equal")
plt.axis("square")
plt.xlim([0, plt.xlim()[1]])
plt.ylim([0, plt.ylim()[1]])

# Plot ideal line - Predicted value equal to Experimental
max_val = max(plt.xlim()[1], plt.ylim()[1])
plt.plot(
    [0, max_val],
    [0, max_val],
    linestyle="--",
    color="gray",
    label="Ideal line",
)

plt.legend(loc="upper left")
plt.show()

Plotting Predicted vs Experimental Values for $\gamma_2$

In [ ]:
# Calculate R²
r2_metric = R2Score()
r2 = r2_metric(test_outputs[:, 1], y_test_tensor[:, 1]).item()

# Convert tensors to CPU and NumPy for plotting
y_test_np = y_test_tensor[:, 1].cpu().numpy()
preds_np = test_outputs[:, 1].detach().cpu().numpy()

# Plot the scatter plot
plt.scatter(y_test_np, preds_np, label=f"R² = {r2:.4f}")
plt.xlabel("Experimental values")
plt.ylabel("Predicted values")
plt.axis("equal")
plt.axis("square")
plt.xlim([0, plt.xlim()[1]])
plt.ylim([0, plt.ylim()[1]])

# Plot ideal line - Predicted value equal to Experimental
max_val = max(plt.xlim()[1], plt.ylim()[1])
plt.plot(
    [0, max_val],
    [0, max_val],
    linestyle="--",
    color="gray",
    label="Ideal line",
)

plt.legend(loc="upper left")
plt.show()

Plotting Gibbs-Duhem Residual Values

In [ ]:
# Convert tensor to NumPy
np_gd_residuals = test_residuals.detach().cpu().numpy()

# Create index for each test sample
test_idxs = np.arange(len(np_gd_residuals))

# Plot residuals against observation index
plt.scatter(test_idxs, np_gd_residuals)
plt.axhline(0, color='gray', linestyle='--')
plt.xlabel("Test Samples")
plt.ylabel("Gibbs-Duhem Residual")
plt.show()